## Cleaning policy

This notebook creates the reusable cleaned dataset used by later stages.

It removes rows without a clear final outcome, removes information unavailable when a loan is approved, creates deterministic features, and preserves remaining missing values for train-only imputation in the modeling pipeline.

In [15]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/FOIA_7a_FY2010_FY2019_asof_260630.csv"
PROCESSED_PATH = "../data/processed/sba_7a_cleaned.csv"

df = pd.read_csv(RAW_PATH, low_memory=False)

df.shape

(545751, 42)

In [16]:
required_columns = {
    "LoanStatus",
    "GrossApproval",
    "SBAGuaranteedApproval",
    "FranchiseCode",
    "NaicsCode",
    "ApprovalDate",
    "SoldSecMrktInd",
}

missing_columns = required_columns - set(df.columns)

assert not missing_columns, (
    f"Raw dataset is missing expected columns: {missing_columns}"
)

In [17]:
clean_df = df[df["LoanStatus"].isin(["P I F", "CHGOFF"])].copy()

clean_df["default"] = clean_df["LoanStatus"].map({
    "P I F": 0,
    "CHGOFF": 1
})

clean_df["default"].value_counts(normalize=True).mul(100).round(2)

default
0    92.04
1     7.96
Name: proportion, dtype: float64

In [18]:
assert clean_df["GrossApproval"].gt(0).all(), (
    "GrossApproval must be positive before calculating guarantee ratios."
)

clean_df["sba_guarantee_ratio"] = (
    clean_df["SBAGuaranteedApproval"] / clean_df["GrossApproval"]
)

assert clean_df["sba_guarantee_ratio"].between(0, 1).all()
assert clean_df["default"].isin([0, 1]).all()

clean_df["is_franchise"] = clean_df["FranchiseCode"].notna().astype(int)

clean_df["naics_sector"] = (
    clean_df["NaicsCode"]
    .astype("Int64")
    .astype(str)
    .str[:2]
)

In [19]:
date_cols = ["ApprovalDate"]

for col in date_cols:
    clean_df[col] = pd.to_datetime(clean_df[col], errors="coerce")

In [20]:
clean_df["approval_month"] = clean_df["ApprovalDate"].dt.month

In [21]:
secondary_market_values = clean_df["SoldSecMrktInd"].value_counts(dropna=False)
display(secondary_market_values)

expected_secondary_market_values = {"Y", "N"}
observed_secondary_market_values = set(clean_df["SoldSecMrktInd"].dropna().unique())

assert observed_secondary_market_values <= expected_secondary_market_values, (
    f"Unexpected SoldSecMrktInd values: {observed_secondary_market_values - expected_secondary_market_values}"
)

SoldSecMrktInd
NaN    324080
Y      104683
N         111
Name: count, dtype: int64

In [22]:
clean_df["sold_secondary_market"] = clean_df["SoldSecMrktInd"].map({"Y": 1, "N": 0})

In [23]:
drop_cols = [
    "LoanStatus",
    "PaidInFullDate",
    "ChargeOffDate",
    "GrossChargeOffAmount",
    "AsOfDate",
    "Program",
    "LocationID",
    "BorrName",
    "BorrStreet",
    "BorrCity",
    "BorrZip",
    "BankName",
    "BankFDICNumber",
    "BankNCUANumber",
    "BankStreet",
    "BankCity",
    "BankZip",
    "FranchiseCode",
    "FranchiseName",
    "NaicsDescription",
    "SoldSecMrktInd",
    "ProjectCounty",
    "ApprovalDate",
    "FirstDisbursementDate",
]

In [24]:
clean_df = clean_df.drop(columns=drop_cols)

clean_df.shape

(428874, 25)

In [25]:
clean_df.loc[clean_df["InitialInterestRate"] == 0, "InitialInterestRate"] = np.nan
clean_df.loc[clean_df["TermInMonths"] == 0, "TermInMonths"] = np.nan

In [26]:
missing_after_drop = (
    clean_df
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_after_drop[missing_after_drop > 0].round(3)

sold_secondary_market    75.565
BusinessAge               0.287
TermInMonths              0.052
CongressionalDistrict     0.006
BusinessType              0.004
InitialInterestRate       0.001
naics_sector              0.000
NaicsCode                 0.000
dtype: float64

In [27]:
numeric_cols = clean_df.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_cols = clean_df.select_dtypes(include=["object", "string"]).columns.tolist()

numeric_cols.remove("default")

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['GrossApproval', 'SBAGuaranteedApproval', 'ApprovalFY', 'InitialInterestRate', 'TermInMonths', 'NaicsCode', 'CongressionalDistrict', 'JobsSupported', 'sba_guarantee_ratio', 'is_franchise', 'sold_secondary_market']
Categorical: ['BorrState', 'BankState', 'ProcessingMethod', 'FixedorVariableInterestInd', 'ProjectState', 'SBADistrictOffice', 'BusinessType', 'BusinessAge', 'RevolverStatus', 'CollateralInd', 'naics_sector']


In [28]:
clean_df.head()

,BorrState,BankState,GrossApproval,SBAGuaranteedApproval,ApprovalFY,ProcessingMethod,InitialInterestRate,FixedorVariableInterestInd,TermInMonths,NaicsCode,...,RevolverStatus,JobsSupported,CollateralInd,default,sba_guarantee_ratio,is_franchise,naics_sector,approval_month,approval_quarter,sold_secondary_market
0,TX,TX,288000.0,259200.0,2010,Preferred Lenders Program,6.00,V,120.0,722110.0,...,N,18.0,Y,0,0.9,0,72,10,4,NaN
1,FL,NC,1200000.0,1080000.0,2010,Preferred Lenders Program,4.75,V,300.0,541940.0,...,N,10.0,N,0,0.9,0,54,10,4,1.0
2,TX,OH,120000.0,108000.0,2010,Preferred Lenders Program,5.25,V,90.0,312113.0,...,N,4.0,N,0,0.9,0,31,10,4,NaN
3,MI,OH,150000.0,75000.0,2010,SBA Express Program,5.25,V,60.0,722211.0,...,N,48.0,Y,0,0.5,1,72,10,4,NaN
5,NY,OH,25000.0,12500.0,2010,SBA Express Program,6.50,V,84.0,238210.0,...,Y,4.0,Y,0,0.5,0,23,10,4,NaN


In [29]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 428874 entries, 0 to 545749
Data columns (total 25 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   BorrState                   428874 non-null  str    
 1   BankState                   428874 non-null  str    
 2   GrossApproval               428874 non-null  float64
 3   SBAGuaranteedApproval       428874 non-null  float64
 4   ApprovalFY                  428874 non-null  int64  
 5   ProcessingMethod            428874 non-null  str    
 6   InitialInterestRate         428869 non-null  float64
 7   FixedorVariableInterestInd  428874 non-null  str    
 8   TermInMonths                428652 non-null  float64
 9   NaicsCode                   428872 non-null  float64
 10  ProjectState                428874 non-null  str    
 11  SBADistrictOffice           428874 non-null  str    
 12  CongressionalDistrict       428849 non-null  float64
 13  BusinessType                42

In [30]:
clean_df["default"].value_counts(normalize=True).mul(100).round(2)

default
0    92.04
1     7.96
Name: proportion, dtype: float64

In [31]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)

In [32]:
reloaded_df = pd.read_csv(PROCESSED_PATH)

assert reloaded_df.shape == clean_df.shape
assert reloaded_df.columns.tolist() == clean_df.columns.tolist()

print("Processed dataset saved and reloaded successfully.")
print(f"Shape: {reloaded_df.shape}")

(428874, 25)

## Cleaning Decisions

### Outcome definition

- Kept only loans with clear final outcomes:
  - `P I F`: paid in full
  - `CHGOFF`: charged off
- Created the binary target `default`:
  - `0`: paid in full
  - `1`: charged off
- Excluded ambiguous statuses such as `CANCLD`, `EXEMPT`, and `COMMIT`.

### Features removed

- Removed post-outcome leakage columns:
  - `LoanStatus`
  - `PaidInFullDate`
  - `ChargeOffDate`
  - `GrossChargeOffAmount`
  - `AsOfDate`
- Removed identity, address, and high-cardinality fields.
- Removed `Program`, since all retained records are SBA 7(a).
- Removed `FirstDisbursementDate`, which may not be known at loan approval.
- Replaced raw franchise and secondary-market fields with simplified features.

### Features created

- `default`
- `sba_guarantee_ratio`
- `is_franchise`
- `naics_sector`
- `approval_month`
- `sold_secondary_market`

### Data-quality handling

- Converted zero values in `InitialInterestRate` and `TermInMonths` to missing values.
- Kept zero values in `JobsSupported`, since they may be valid.
- Validated that gross approval amounts are positive and guarantee ratios fall between 0 and 1.
- Validated that non-missing `SoldSecMrktInd` values are limited to `Y` and `N`.
- Preserved remaining missing values for imputation within the training-only modeling pipeline.